# Genome-wide methylation maps (pb-CpG-tools)

Per-sample pb-CpG-tools stats from the `aou2_v1_phased_bams` pileups.

The sample table lives in the storage workspace
`allofus-drc-wgs-LR-prodData` / `AoU_DRC_LongReads_PhaseTwo_Storage`.
This notebook pulls it with firecloud (`get_entities_tsv`), then
downloads each row's `stats_tsv` (written back by `PbCpgSampleStats.wdl`).

**Terra:** `edit/scripts/` on the VM is often stale. From a current git
checkout, stage CLIs to the workspace bucket, then re-run the setup cell
(it rsyncs `$WORKSPACE_BUCKET/scripts/` locally before importing):

```bash
gsutil -m rsync -r scripts/ "$WORKSPACE_BUCKET/scripts/"
```

Override with `METH_TABLE_TSV` only if you already exported the table.
Set `METH_RUN_PIPELINE=true` before downloading stats / merging.

## Outputs

- `summaries/manuscript/pbcpg_table_inventory.json`
- `summaries/manuscript/pbcpg_table_inventory.table.tsv`
- `summaries/manuscript/pbcpg_sample_stats.tsv`
- `summaries/manuscript/methylation_manuscript_numbers.{tsv,json}`
- `summaries/manuscript/methylation_concordance.json` (optional)
- `summaries/manuscript/methylation_concordance.pairs.tsv.gz` (optional)
- `summaries/manuscript/figS3_methylation.{png,pdf}` (optional)


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# On Terra, localize $WORKSPACE_BUCKET/scripts/ before importing anything.
# Persistent edit/scripts/ copies are often stale and must not win.
_bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
_sync = os.environ.get("TERRA_SYNC_SCRIPTS", "true" if _bucket else "").strip().lower() in {
    "1", "true", "yes", "on",
}
_scripts = None
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file() or (_d / "workspace_paths.py").is_file():
        _scripts = _d.resolve()
        break
if _bucket and _sync:
    if _scripts is None:
        _scripts = (Path.cwd() / "scripts").resolve()
    _scripts.mkdir(parents=True, exist_ok=True)
    print(f"gsutil -m rsync -r {_bucket}/scripts/ {_scripts}/")
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_scripts) + "/"])
    os.environ["TERRA_SCRIPTS_LOCALIZED"] = "true"
    import importlib
    importlib.invalidate_caches()
    _prefix = str(_scripts)
    for _name, _mod in list(sys.modules.items()):
        _file = getattr(_mod, "__file__", None)
        if _file and str(_file).startswith(_prefix):
            sys.modules.pop(_name, None)
elif _scripts is None:
    raise FileNotFoundError(
        "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
        "Upload scripts/ to gs://WORKSPACE/scripts/."
    )
sys.path.insert(0, str(_scripts))

from terra_notebook import init_notebook

SCRIPTS = init_notebook("workspace_paths.py", "pbcpg_stats.py")
from workspace_paths import data_root
try:
    from workspace_paths import methylation_output_dir
except ImportError as exc:
    raise ImportError(
        f"{getattr(sys.modules.get('workspace_paths'), '__file__', 'workspace_paths')} is stale "
        "(no methylation_output_dir). From a current git checkout run "
        'gsutil -m rsync -r scripts/ "$WORKSPACE_BUCKET/scripts/" '
        "and re-run this cell."
    ) from exc
from pbcpg_stats import (
    DEFAULT_ENTITY_TYPE,
    DEFAULT_ID_COLUMN,
    DEFAULT_NAMESPACE,
    DEFAULT_WORKSPACE,
    fetch_phased_bams_table,
    inventory_from_frame,
    pull_stats_from_table,
)
import json
import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

ROOT = data_root()
WORKSPACE_BUCKET = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
SUMMARY_DIR = Path(os.environ.get("METH_SUMMARY_DIR", ROOT / "summaries" / "manuscript"))
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
STATS_DIR = Path(os.environ.get("METH_STATS_DIR", methylation_output_dir() / "shards"))
DUMPS_DIR = Path(os.environ.get("METH_DUMPS_DIR", methylation_output_dir() / "chr22_dumps"))
TABLE_TSV = os.environ.get("METH_TABLE_TSV", "")
COV_CSV = Path(
    os.environ.get(
        "AOU_COVARIATES",
        ROOT / "covariates.source_rebuilt.csv.gz",
    )
)
if not COV_CSV.is_file():
    alt = Path.cwd() / "covariates.v6.csv.gz"
    if alt.is_file():
        COV_CSV = alt

RUN_PIPELINE = os.environ.get("METH_RUN_PIPELINE", "").lower() in {"1", "true", "yes", "on"}
RUN_INVENTORY = os.environ.get("METH_RUN_INVENTORY", "true").lower() in {"1", "true", "yes", "on"}
RUN_MERGE = os.environ.get("METH_RUN_MERGE", "true").lower() in {"1", "true", "yes", "on"}
RUN_CONCORDANCE = os.environ.get("METH_RUN_CONCORDANCE", "").lower() in {"1", "true", "yes", "on"}
DISCOVERY_ONLY = os.environ.get("METH_DISCOVERY_ONLY", "true").lower() in {"1", "true", "yes", "on"}
TERRA_NAMESPACE = os.environ.get("METH_TERRA_NAMESPACE", DEFAULT_NAMESPACE)
TERRA_WORKSPACE = os.environ.get("METH_TERRA_WORKSPACE", DEFAULT_WORKSPACE)
ENTITY_TYPE = os.environ.get("METH_ENTITY_TYPE", DEFAULT_ENTITY_TYPE)
PULL_JOBS = int(os.environ.get("METH_PULL_JOBS", "8"))

print("ROOT:", ROOT)
print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET or "(local)")
print("SUMMARY_DIR:", SUMMARY_DIR)
print("STATS_DIR:", STATS_DIR)
print("DUMPS_DIR:", DUMPS_DIR)
print("COV_CSV:", COV_CSV, "exists=" + str(COV_CSV.is_file()))
print("RUN_PIPELINE:", RUN_PIPELINE)
print("RUN_INVENTORY:", RUN_INVENTORY)
print("RUN_MERGE:", RUN_MERGE)
print("RUN_CONCORDANCE:", RUN_CONCORDANCE)
print("TERRA:", f"{TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}")
print("PULL_JOBS:", PULL_JOBS)


## 1. Fetch `aou2_v1_phased_bams`

Uses firecloud against `AoU_DRC_LongReads_PhaseTwo_Storage` unless
`METH_TABLE_TSV` points at an export. This cell does not need
`METH_RUN_PIPELINE`.


In [ ]:
table_json = SUMMARY_DIR / "pbcpg_table_inventory.json"
table_tsv_out = SUMMARY_DIR / "pbcpg_table_inventory.table.tsv"

if TABLE_TSV:
    print("table TSV:", TABLE_TSV)
else:
    print(f"firecloud get_entities_tsv {TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}")

phased_bams = fetch_phased_bams_table(
    tsv=TABLE_TSV or None,
    from_firecloud=not TABLE_TSV,
    namespace=TERRA_NAMESPACE,
    workspace=TERRA_WORKSPACE,
    entity_type=ENTITY_TYPE,
    id_column=DEFAULT_ID_COLUMN,
)
id_column = DEFAULT_ID_COLUMN if DEFAULT_ID_COLUMN in phased_bams.columns else [
    c for c in phased_bams.columns if c.startswith("entity:") or c.endswith("_id")
][0]
inventory = inventory_from_frame(phased_bams, id_column=id_column)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
phased_bams.to_csv(table_tsv_out, sep="\t", index=False)
table_json.write_text(json.dumps(inventory, indent=2, sort_keys=True) + "\n")
print(f"rows={len(phased_bams):,} columns={len(phased_bams.columns)}")
print("id column:", id_column)
print("columns:", ", ".join(map(str, phased_bams.columns[:25])), "..." if len(phased_bams.columns) > 25 else "")
display(inventory)
display(phased_bams.head())


## 2. Pull `stats_tsv` from the table and merge

After `PbCpgSampleStats.wdl` writes outputs back to `aou2_v1_phased_bams`,
each row has a `stats_tsv` GCS URI. This copies them to `STATS_DIR` as
`{research_id}.tsv` and merges. Set `METH_RUN_PIPELINE=true`.


In [ ]:
n_local = len(list(STATS_DIR.rglob("*.tsv"))) if STATS_DIR.is_dir() else 0
print("local stats TSVs:", n_local, "in", STATS_DIR)
print("table stats_tsv URIs:", inventory.get("n_stats_tsv"), "column=", inventory.get("stats_uri_column") or "(none)")

merge_cmd = [
    "merge",
    "--stats-dir", str(STATS_DIR),
    "--out-dir", str(SUMMARY_DIR),
]
if COV_CSV.is_file():
    merge_cmd.extend(["--covariates", str(COV_CSV)])
if DISCOVERY_ONLY:
    merge_cmd.append("--discovery-only")

from pbcpg_stats import main as pbcpg_main

print("merge:", " ".join(merge_cmd))
if RUN_PIPELINE and RUN_MERGE:
    if inventory.get("n_stats_tsv"):
        pull = pull_stats_from_table(
            phased_bams,
            STATS_DIR,
            id_column=id_column,
            jobs=PULL_JOBS,
            force=os.environ.get("METH_FORCE", "").lower() in {"1", "true", "yes", "on"},
        )
        display(pull)
    elif n_local == 0:
        raise FileNotFoundError(
            "No stats_tsv on the data table and none in STATS_DIR. "
            "Submit PbCpgSampleStats.wdl with outputs written to aou2_v1_phased_bams."
        )
    pbcpg_main(merge_cmd)
    display(json.loads((SUMMARY_DIR / "methylation_manuscript_numbers.json").read_text()))
    display(pd.read_csv(SUMMARY_DIR / "methylation_manuscript_numbers.tsv", sep="\t"))
else:
    print("dry-run: set METH_RUN_PIPELINE=true to pull stats_tsv and merge")


## 3. Primrose vs Jasmine concordance

Fetches `aou2_v1_phased_bams`, downloads a Primrose + Jasmine subsample of
the `sites` column into `DUMPS_DIR`, then computes concordance.
Set `METH_RUN_CONCORDANCE=true`.


In [ ]:
from pbcpg_stats import (
    CONCORDANCE_N_PER_GROUP,
    fetch_phased_bams_table,
    load_site_dump,
    pull_dumps_from_table,
    sites_uri_column,
)
from pbcpg_stats import main as pbcpg_main

N_PER_GROUP = int(os.environ.get("METH_CONCORDANCE_N", str(CONCORDANCE_N_PER_GROUP)))
DUMPS_DIR.mkdir(parents=True, exist_ok=True)
labels_path = SUMMARY_DIR / "pbcpg_concordance_labels.tsv"
conc_json = SUMMARY_DIR / "methylation_concordance.json"
stats_tsv = SUMMARY_DIR / "pbcpg_sample_stats.tsv"
stats_arg = str(stats_tsv) if stats_tsv.is_file() else (str(STATS_DIR) if STATS_DIR.is_dir() else None)

if TABLE_TSV:
    print("table TSV:", TABLE_TSV)
else:
    print(f"firecloud get_entities_tsv {TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}")

sites_table = fetch_phased_bams_table(
    tsv=TABLE_TSV or None,
    from_firecloud=not TABLE_TSV,
    namespace=TERRA_NAMESPACE,
    workspace=TERRA_WORKSPACE,
    entity_type=ENTITY_TYPE,
    id_column=DEFAULT_ID_COLUMN,
)
sites_id = DEFAULT_ID_COLUMN if DEFAULT_ID_COLUMN in sites_table.columns else [
    c for c in sites_table.columns if str(c).startswith("entity:") or str(c).endswith("_id")
][0]
sites_col = sites_uri_column(sites_table)
n_sites = int(sites_table[sites_col].astype(str).str.startswith("gs://").sum()) if sites_col else 0
print("id column:", sites_id)
print("sites column:", sites_col or "(none)", "gs:// rows:", n_sites)

if not RUN_CONCORDANCE:
    print("dry-run: set METH_RUN_CONCORDANCE=true to pull sites dumps and compute concordance")
else:
    if not sites_col or n_sites == 0:
        raise FileNotFoundError(
            f"No sites URIs on {TERRA_NAMESPACE}/{TERRA_WORKSPACE}/{ENTITY_TYPE}. "
            f"Columns: {list(sites_table.columns)}"
        )
    if not COV_CSV.is_file():
        raise FileNotFoundError(f"Need covariates for pb_meth_caller: {COV_CSV}")
    pull = pull_dumps_from_table(
        sites_table,
        DUMPS_DIR,
        id_column=sites_id,
        covariates=str(COV_CSV),
        stats=stats_arg,
        n_per_group=N_PER_GROUP,
        jobs=PULL_JOBS,
        force=os.environ.get("METH_FORCE", "").lower() in {"1", "true", "yes", "on"},
    )
    display(pull)
    sample_ids = []
    for path in DUMPS_DIR.iterdir():
        if path.is_file() and (path.suffix in {".gz", ".tsv"} or path.name.endswith(".tsv.gz")):
            sample_ids.append(load_site_dump(path)[0])
    cov = pd.read_csv(COV_CSV, dtype={"research_id": str}, low_memory=False)
    labels = cov.loc[cov["research_id"].isin(sample_ids), ["research_id", "pb_meth_caller"]].rename(
        columns={"research_id": "sample_id"}
    )
    labels.to_csv(labels_path, sep="\t", index=False)
    pbcpg_main(
        [
            "concordance",
            "--dumps-dir", str(DUMPS_DIR),
            "--labels", str(labels_path),
            "--out", str(conc_json),
        ]
    )
    display(json.loads(conc_json.read_text()))


## 4. Fig S3 data panels

Hexbin of Primrose vs Jasmine **site-mean** 5mC (not paired samples) plus
haplotype-slot occupancy vs coverage from `pbcpg_sample_stats.tsv`.

Uses dumps already pulled for concordance, or
`methylation_concordance.pairs.tsv.gz` if that exists. Set
`METH_RUN_PLOT=true` (or re-run concordance; it now writes the pairs file).
Copy `summaries/manuscript/figS3_methylation.png` into the manuscript
`figures/` directory when replacing the mock-up B/D panels.


In [ ]:
from IPython.display import Image, display
from pbcpg_stats import main as pbcpg_main

pairs_path = SUMMARY_DIR / "methylation_concordance.pairs.tsv.gz"
labels_path = SUMMARY_DIR / "pbcpg_concordance_labels.tsv"
stats_tsv = SUMMARY_DIR / "pbcpg_sample_stats.tsv"
fig_prefix = SUMMARY_DIR / "figS3_methylation"

run_plot = RUN_PLOT or RUN_CONCORDANCE or pairs_path.is_file() or any(DUMPS_DIR.glob("*.tsv.gz"))
if not run_plot:
    print("dry-run: set METH_RUN_PLOT=true after concordance dumps exist")
else:
    plot_args = ["plot-s3", "--out-prefix", str(fig_prefix)]
    if stats_tsv.is_file():
        plot_args += ["--stats", str(stats_tsv)]
    if pairs_path.is_file():
        plot_args += ["--pairs", str(pairs_path)]
    elif DUMPS_DIR.is_dir() and labels_path.is_file():
        plot_args += [
            "--dumps-dir", str(DUMPS_DIR),
            "--labels", str(labels_path),
            "--write-pairs", str(pairs_path),
        ]
    else:
        raise FileNotFoundError(
            "Need methylation_concordance.pairs.tsv.gz or chr22 dumps + labels. "
            "Run concordance first (METH_RUN_CONCORDANCE=true)."
        )
    pbcpg_main(plot_args)
    png = Path(str(fig_prefix) + ".png")
    if png.is_file():
        display(Image(filename=str(png)))
